# Exploratory End-to-End Machine Learning Pipeline in Python

This Jupyter notebook is a visual, commented walkthrough of the complete ML lifecycle:

1. Library map  
2. Interactive data selection  
3. Ingestion  
4. Validation and profiling  
5. Cleaning  
6. Visualization  
7. Feature engineering  
8. Train/validation/test splitting  
9. Preprocessing  
10. Training  
11. Hyperparameter tuning  
12. Evaluation  
13. Explainability  
14. Model serialization  
15. Batch inference  
16. API serving concepts  
17. Monitoring and retraining concepts  

You may either:

- Press **Enter** to use the built-in sample data.
- Paste a small CSV dataset.
- Paste a small JSON array of records.

Keyboard input is capped at **1,000 characters**.

## 1. Main Python libraries across the ML pipeline

| Stage | Libraries |
|---|---|
| Data ingestion | `pandas`, `pyarrow`, `sqlalchemy`, `requests`, `boto3` |
| Validation | `pandera`, `great_expectations`, `pydantic` |
| Cleaning | `pandas`, `numpy`, `scipy` |
| Visualization | `matplotlib`, `seaborn`, `plotly`, `missingno` |
| Feature engineering | `pandas`, `scikit-learn`, `feature-engine` |
| Training | `scikit-learn`, `xgboost`, `lightgbm`, `catboost`, `torch`, `tensorflow` |
| Tuning | `GridSearchCV`, `RandomizedSearchCV`, `optuna`, `ray[tune]` |
| Explainability | `shap`, `lime`, `interpret` |
| Tracking | `mlflow`, `wandb` |
| Serialization | `joblib`, `pickle`, `onnx`, `skops` |
| Serving | `fastapi`, `uvicorn`, `bentoml` |
| Monitoring | `evidently`, `whylogs`, `prometheus_client` |
| Orchestration | `airflow`, `prefect`, `dagster`, `kubeflow` |

In [ ]:
from __future__ import annotations

from io import StringIO
from pathlib import Path
from typing import Literal
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
MAX_KEYBOARD_CHARACTERS = 1_000

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
OUTPUT_DIR = Path("outputs")

for directory in [DATA_DIR, MODEL_DIR, OUTPUT_DIR]:
    directory.mkdir(exist_ok=True)

print("Notebook environment ready.")

## 2. Interactive data selection

Choose:

- `1` or blank: built-in default dataset
- `2`: paste CSV
- `3`: paste JSON records

### Small CSV example

```csv
usage_score,support_calls,billing_variance,engagement_score,tenure_signal,risk_signal,customer_id,region,plan_type,monthly_spend,signup_date,churned
0.2,1,0.1,0.8,1.2,-0.3,CUST-1,North,Basic,59.5,2024-01-01,0
-0.4,4,0.6,-0.5,0.3,1.1,CUST-2,West,Premium,105.0,2023-04-12,1
```

### Small JSON example

```json
[{"usage_score":0.2,"support_calls":1,"billing_variance":0.1,"engagement_score":0.8,"tenure_signal":1.2,"risk_signal":-0.3,"customer_id":"CUST-1","region":"North","plan_type":"Basic","monthly_spend":59.5,"signup_date":"2024-01-01","churned":0}]
```

In [ ]:
def create_default_dataset(rows: int = 2_000) -> pd.DataFrame:
    X_raw, y_raw = make_classification(
        n_samples=rows,
        n_features=6,
        n_informative=4,
        n_redundant=1,
        weights=[0.72, 0.28],
        class_sep=1.1,
        random_state=RANDOM_STATE,
    )

    rng = np.random.default_rng(RANDOM_STATE)

    data = pd.DataFrame(
        X_raw,
        columns=[
            "usage_score",
            "support_calls",
            "billing_variance",
            "engagement_score",
            "tenure_signal",
            "risk_signal",
        ],
    )

    data["customer_id"] = [
        f"CUST-{i:05d}" for i in range(len(data))
    ]

    data["region"] = rng.choice(
        ["North", "South", "East", "West"],
        size=len(data),
    )

    data["plan_type"] = rng.choice(
        ["Basic", "Standard", "Premium"],
        size=len(data),
        p=[0.45, 0.40, 0.15],
    )

    data["monthly_spend"] = (
        55
        + data["usage_score"] * 8
        + rng.normal(0, 12, size=len(data))
    ).clip(5, 250)

    data["signup_date"] = (
        pd.Timestamp("2021-01-01")
        + pd.to_timedelta(
            rng.integers(0, 1_500, size=len(data)),
            unit="D",
        )
    )

    data["churned"] = y_raw

    # Add realistic quality problems.
    missing_rows = rng.choice(data.index, size=120, replace=False)
    data.loc[missing_rows[:40], "monthly_spend"] = np.nan
    data.loc[missing_rows[40:80], "region"] = None
    data.loc[missing_rows[80:], "engagement_score"] = np.nan

    data.loc[
        rng.choice(data.index, size=25, replace=False),
        "plan_type",
    ] = "premium"

    data.loc[
        rng.choice(data.index, size=20, replace=False),
        "region",
    ] = "WEST"

    data.loc[
        rng.choice(data.index, size=10, replace=False),
        "monthly_spend",
    ] = -50

    duplicates = data.sample(15, random_state=RANDOM_STATE)

    return pd.concat([data, duplicates], ignore_index=True)


def parse_keyboard_data(
    input_format: Literal["csv", "json"],
    raw_text: str,
) -> pd.DataFrame:
    if len(raw_text) > MAX_KEYBOARD_CHARACTERS:
        raise ValueError(
            f"Input exceeds the {MAX_KEYBOARD_CHARACTERS:,}-character limit."
        )

    if input_format == "csv":
        return pd.read_csv(StringIO(raw_text))

    parsed = json.loads(raw_text)

    if not isinstance(parsed, list):
        raise ValueError(
            "JSON input must be an array of record objects."
        )

    return pd.DataFrame(parsed)


def choose_input_data() -> pd.DataFrame:
    print("Select a data source:")
    print("  1 or blank = built-in default dataset")
    print("  2          = paste CSV")
    print("  3          = paste JSON")
    print()

    choice = input("Choice [default=1]: ").strip().lower()

    if choice in {"", "1", "default", "d"}:
        print("Using the built-in dataset.")
        return create_default_dataset()

    if choice not in {"2", "3", "csv", "json"}:
        print("Unknown choice. Using the built-in dataset.")
        return create_default_dataset()

    input_format: Literal["csv", "json"] = (
        "csv" if choice in {"2", "csv"} else "json"
    )

    print()
    print(
        f"Paste up to {MAX_KEYBOARD_CHARACTERS:,} characters "
        f"of {input_format.upper()}."
    )
    print(
        "Press Enter without data to fall back to defaults."
    )

    raw_text = input(
        f"{input_format.upper()} data: "
    ).strip()

    if not raw_text:
        print("No data supplied. Using the built-in dataset.")
        return create_default_dataset()

    try:
        keyboard_df = parse_keyboard_data(
            input_format=input_format,
            raw_text=raw_text,
        )

        if keyboard_df.empty:
            raise ValueError("The pasted dataset has no rows.")

        print(
            f"Loaded {len(keyboard_df):,} rows and "
            f"{len(keyboard_df.columns):,} columns."
        )

        return keyboard_df

    except Exception as exc:
        print("Could not parse pasted data:")
        print(exc)
        print("Using the built-in dataset instead.")
        return create_default_dataset()


df = choose_input_data()

print()
print("Selected dataset shape:", df.shape)
display(df.head())

> **Schema note**
>
> You may paste any small CSV or JSON dataset for ingestion and inspection.
>
> The later training cells expect the default churn schema. To run them unchanged, use the default data or map your own columns to:
>
> `usage_score`, `support_calls`, `billing_variance`, `engagement_score`,
> `tenure_signal`, `risk_signal`, `customer_id`, `region`, `plan_type`,
> `monthly_spend`, `signup_date`, and `churned`.

## 3. Persist and ingest the selected data

This creates a real file boundary and then reloads the data with `pandas`.

In [ ]:
raw_path = DATA_DIR / "selected_input_data.csv"

df.to_csv(raw_path, index=False)

ingested_df = pd.read_csv(raw_path)

if "signup_date" in ingested_df.columns:
    ingested_df["signup_date"] = pd.to_datetime(
        ingested_df["signup_date"],
        errors="coerce",
    )

print("Saved source:", raw_path.resolve())
print("Ingested shape:", ingested_df.shape)
display(ingested_df.head())

## 4. Profile and validate the data

This stage checks:

- Types
- Missing values
- Duplicate records
- Unique values
- Expected columns

In [ ]:
def profile_dataframe(data: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "missing_count": data.isna().sum(),
        "missing_percent": (data.isna().mean() * 100).round(2),
        "unique_values": data.nunique(dropna=False),
    }).sort_values("missing_percent", ascending=False)


display(profile_dataframe(ingested_df))

print("Duplicate rows:", ingested_df.duplicated().sum())

In [ ]:
REQUIRED_COLUMNS = {
    "usage_score",
    "support_calls",
    "billing_variance",
    "engagement_score",
    "tenure_signal",
    "risk_signal",
    "customer_id",
    "region",
    "plan_type",
    "monthly_spend",
    "signup_date",
    "churned",
}


def has_training_schema(data: pd.DataFrame) -> bool:
    missing = REQUIRED_COLUMNS - set(data.columns)

    if missing:
        print("Training schema is incomplete.")
        print("Missing columns:", sorted(missing))
        return False

    print("Training schema is available.")
    return True


TRAINING_SCHEMA_AVAILABLE = has_training_schema(ingested_df)

## 5. Visualize raw data quality

In [ ]:
missing_percent = (
    ingested_df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))

nonzero_missing = missing_percent[missing_percent > 0]

if len(nonzero_missing):
    nonzero_missing.plot(kind="bar", ax=ax)
    ax.set_title("Missing values by column")
    ax.set_ylabel("Missing values (%)")
    ax.tick_params(axis="x", rotation=45)
else:
    ax.text(
        0.5,
        0.5,
        "No missing values found",
        ha="center",
        va="center",
        fontsize=14,
    )
    ax.set_axis_off()

fig.tight_layout()
plt.show()

## 6. Clean and structure the data

The training branch runs only when the expected schema is present.

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    clean_df = ingested_df.copy()

    clean_df = clean_df.drop_duplicates()

    clean_df["region"] = (
        clean_df["region"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    clean_df["plan_type"] = (
        clean_df["plan_type"]
        .astype("string")
        .str.strip()
        .str.title()
    )

    clean_df.loc[
        clean_df["monthly_spend"] < 0,
        "monthly_spend",
    ] = np.nan

    clean_df["churned"] = clean_df["churned"].astype(int)

    print("Cleaned shape:", clean_df.shape)
    display(clean_df.head())
else:
    clean_df = ingested_df.copy()
    print(
        "Custom data was profiled, but model training is skipped "
        "until its columns are mapped to the expected schema."
    )

## 7. Feature engineering

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    feature_df = clean_df.copy()

    reference_date = pd.Timestamp("2026-01-01")

    feature_df["tenure_days"] = (
        reference_date - feature_df["signup_date"]
    ).dt.days

    feature_df["spend_per_tenure_year"] = (
        feature_df["monthly_spend"]
        / np.maximum(
            feature_df["tenure_days"] / 365.25,
            0.25,
        )
    )

    feature_df["signup_month"] = (
        feature_df["signup_date"].dt.month
    )

    feature_df["signup_year"] = (
        feature_df["signup_date"].dt.year
    )

    model_df = feature_df.drop(
        columns=["customer_id", "signup_date"]
    )

    display(model_df.head())

## 8. Exploratory visualization

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    churn_by_plan = (
        model_df.groupby("plan_type", dropna=False)["churned"]
        .mean()
        .sort_values(ascending=False)
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    churn_by_plan.plot(kind="bar", ax=ax)
    ax.set_title("Average churn rate by plan")
    ax.set_xlabel("Plan")
    ax.set_ylabel("Churn rate")
    ax.tick_params(axis="x", rotation=0)
    fig.tight_layout()
    plt.show()

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    numeric_data = model_df.select_dtypes(include=np.number)
    corr = numeric_data.corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(corr, aspect="auto")
    ax.set_title("Numerical feature correlation matrix")
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticklabels(corr.columns)
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    plt.show()

## 9. Split features and target

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    X = model_df.drop(columns=["churned"])
    y = model_df["churned"]

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y_train_full,
    )

    print("Training rows:", len(X_train))
    print("Validation rows:", len(X_valid))
    print("Test rows:", len(X_test))

## 10. Build preprocessing and model pipelines

Numerical features:

- Median imputation
- Standard scaling

Categorical features:

- Most-frequent imputation
- One-hot encoding

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    numeric_features = (
        X.select_dtypes(include=np.number)
        .columns
        .tolist()
    )

    categorical_features = (
        X.select_dtypes(exclude=np.number)
        .columns
        .tolist()
    )

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])

    preprocessor = ColumnTransformer([
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ])

    print("Numerical features:", numeric_features)
    print("Categorical features:", categorical_features)

## 11. Train a baseline model

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    baseline_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2_000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ])

    baseline_pipeline.fit(X_train, y_train)

    validation_probabilities = (
        baseline_pipeline.predict_proba(X_valid)[:, 1]
    )

    validation_auc = roc_auc_score(
        y_valid,
        validation_probabilities,
    )

    print(
        f"Baseline validation ROC AUC: "
        f"{validation_auc:.3f}"
    )

## 12. Evaluate the baseline visually

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_estimator(
        baseline_pipeline,
        X_valid,
        y_valid,
        ax=ax,
    )
    ax.set_title("Baseline confusion matrix")
    fig.tight_layout()
    plt.show()

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    fig, ax = plt.subplots(figsize=(7, 5))
    RocCurveDisplay.from_estimator(
        baseline_pipeline,
        X_valid,
        y_valid,
        ax=ax,
    )
    ax.set_title("Baseline ROC curve")
    fig.tight_layout()
    plt.show()

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    fig, ax = plt.subplots(figsize=(7, 5))
    PrecisionRecallDisplay.from_estimator(
        baseline_pipeline,
        X_valid,
        y_valid,
        ax=ax,
    )
    ax.set_title("Baseline precision-recall curve")
    fig.tight_layout()
    plt.show()

## 13. Train and tune a random forest

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    forest_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ])

    parameter_distributions = {
        "model__n_estimators": [150, 250, 400],
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2", None],
    }

    search = RandomizedSearchCV(
        forest_pipeline,
        parameter_distributions,
        n_iter=10,
        scoring="roc_auc",
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )

    search.fit(X_train_full, y_train_full)

    best_pipeline = search.best_estimator_

    print(
        "Best cross-validation ROC AUC:",
        round(search.best_score_, 3),
    )
    print("Best parameters:")
    print(json.dumps(search.best_params_, indent=2))

## 14. Final test evaluation

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    test_probabilities = (
        best_pipeline.predict_proba(X_test)[:, 1]
    )

    test_predictions = best_pipeline.predict(X_test)

    test_auc = roc_auc_score(
        y_test,
        test_probabilities,
    )

    print(f"Final test ROC AUC: {test_auc:.3f}")
    print()
    print(
        classification_report(
            y_test,
            test_predictions,
            target_names=["No churn", "Churn"],
        )
    )

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    fig, ax = plt.subplots(figsize=(7, 5))
    RocCurveDisplay.from_estimator(
        best_pipeline,
        X_test,
        y_test,
        ax=ax,
    )
    ax.set_title("Final test ROC curve")
    fig.tight_layout()
    plt.show()

## 15. Explainability with permutation importance

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    importance = permutation_importance(
        best_pipeline,
        X_test,
        y_test,
        scoring="roc_auc",
        n_repeats=6,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    importance_df = pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    }).sort_values("importance_mean")

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(
        importance_df["feature"],
        importance_df["importance_mean"],
        xerr=importance_df["importance_std"],
    )
    ax.set_title("Permutation feature importance")
    ax.set_xlabel("Mean decrease in ROC AUC")
    fig.tight_layout()
    plt.show()

    display(
        importance_df.sort_values(
            "importance_mean",
            ascending=False,
        )
    )

## 16. Save and reload the complete pipeline

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    model_path = MODEL_DIR / "customer_churn_pipeline.joblib"

    joblib.dump(best_pipeline, model_path)

    loaded_pipeline = joblib.load(model_path)

    print("Saved model:", model_path.resolve())
    print("Model size:", model_path.stat().st_size, "bytes")

## 17. Batch inference

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    batch_input = X_test.head(20).copy()

    batch_output = batch_input.copy()
    batch_output["predicted_churn"] = (
        loaded_pipeline.predict(batch_input)
    )
    batch_output["churn_probability"] = (
        loaded_pipeline.predict_proba(batch_input)[:, 1]
    )

    display(
        batch_output[
            ["predicted_churn", "churn_probability"]
        ].head(10)
    )

## 18. Simple FastAPI serving pattern

```python
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

app = FastAPI()
model = joblib.load("models/customer_churn_pipeline.joblib")

class PredictionRequest(BaseModel):
    usage_score: float
    support_calls: float
    billing_variance: float
    engagement_score: float | None = None
    tenure_signal: float
    risk_signal: float
    region: str | None = None
    plan_type: str | None = None
    monthly_spend: float | None = None
    tenure_days: int
    spend_per_tenure_year: float | None = None
    signup_month: int
    signup_year: int

@app.post("/predict")
def predict(request: PredictionRequest):
    frame = pd.DataFrame([request.model_dump()])
    probability = model.predict_proba(frame)[0, 1]
    prediction = model.predict(frame)[0]

    return {
        "prediction": int(prediction),
        "churn_probability": float(probability),
    }
```

Run with:

```bash
uvicorn app:app --reload
```

## 19. Monitoring and retraining

After deployment, monitor:

- Data drift
- Missing values
- Schema changes
- Prediction confidence
- Accuracy when labels arrive
- API latency
- Error rates
- CPU and memory

Useful libraries include:

- `evidently`
- `whylogs`
- `prometheus_client`
- `scipy`
- `pandas`

In [ ]:
if TRAINING_SCHEMA_AVAILABLE:
    train_means = X_train[numeric_features].mean()
    test_means = X_test[numeric_features].mean()
    train_stds = (
        X_train[numeric_features]
        .std()
        .replace(0, np.nan)
    )

    standardized_shift = (
        (test_means - train_means).abs()
        / train_stds
    ).sort_values()

    fig, ax = plt.subplots(figsize=(9, 6))
    standardized_shift.plot(kind="barh", ax=ax)
    ax.set_title("Simple standardized feature shift")
    ax.set_xlabel(
        "Absolute mean shift in training standard deviations"
    )
    fig.tight_layout()
    plt.show()

## 20. Full lifecycle summary

```text
Input files / SQL / APIs / object storage
    ↓
pandas / PyArrow / SQLAlchemy / boto3
    ↓
Pandera / Great Expectations / Pydantic
    ↓
pandas / NumPy cleaning
    ↓
Matplotlib / Plotly visualization
    ↓
Feature engineering
    ↓
scikit-learn preprocessing pipeline
    ↓
Model training
    ↓
RandomizedSearchCV / Optuna
    ↓
Metrics and explainability
    ↓
MLflow experiment tracking
    ↓
joblib / ONNX serialization
    ↓
FastAPI / BentoML serving
    ↓
Evidently / Prometheus monitoring
    ↓
Airflow / Prefect / cloud orchestration
```

The next logical step is to map every one of these stages to AWS services and decide which portions should remain Python and which should become managed AWS infrastructure.